In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

# 1. Memuat Data
df = pd.read_csv('data_hasil_perbaikan/data_preprocessed.csv')

# 2. Preprocessing
# Mengubah target 'Stock' (In Stock / Out of Stock) menjadi numerik
le = LabelEncoder()
df['Stock_Label'] = le.fit_transform(df['Stock']) 

# Mengubah variabel kategori 'Category' menjadi dummy variables (One-Hot Encoding)
df_encoded = pd.get_dummies(df, columns=['Category'], drop_first=True)

# Menentukan Fitur (X) dan Target (y)
X = df_encoded.drop(['Stock', 'Stock_Label'], axis=1)
y = df_encoded['Stock_Label']

# 3. Membagi Data (Train-Test Split)
# 80% untuk pelatihan, 20% untuk pengujian
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Pelatihan Model Random Forest
# Menggunakan 100 pohon keputusan (n_estimators)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 5. Prediksi dan Evaluasi
y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Akurasi Model: {accuracy:.2f}")
print("\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# 6. Visualisasi Feature Importance
importances = rf_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Fitur Paling Berpengaruh terhadap Ketersediaan Stok')
plt.xlabel('Tingkat Kepentingan')
plt.ylabel('Fitur')
plt.tight_layout()
plt.show()

# 7. Matriks Kebingungan (Confusion Matrix)
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix - Prediksi Stok')
plt.show()